<h3>Global alignment, Needleman-Wynsch</h3>

In [1]:
# Installing the packages uncomment
#!pip install pyMSAviz
#!pip install biopython

<a href = "https://www.youtube.com/watch?v=VWzXQhHoepI">Explanation how the algoritm works</a>

<h3>Implementation of the Needleman–Wunsch in Python</h3>

In [12]:
import numpy as np
def needleman_wunsch(seq1, seq2, match_score, mismatch_penalty, gap_penalty):
    # Initialize the score matrix with zeros
    n = len(seq1)
    m = len(seq2)
    score_matrix = np.zeros((n + 1, m + 1)) # constructing the empty matrix

    # Initialize the gap penalties in the score matrix
    for i in range(1, n + 1):
        score_matrix[i][0] = score_matrix[i - 1][0] + gap_penalty
    for j in range(1, m + 1):
        score_matrix[0][j] = score_matrix[0][j - 1] + gap_penalty

    # Fill the score matrix
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            match = score_matrix[i - 1][j - 1] + (match_score if seq1[i - 1] == seq2[j - 1] else mismatch_penalty)
            delete = score_matrix[i - 1][j] + gap_penalty
            insert = score_matrix[i][j - 1] + gap_penalty
            score_matrix[i][j] = max(match, delete, insert)

    # Traceback to find the alignment
    align1 = ""
    align2 = ""
    i = n
    j = m

    while i > 0 or j > 0:
        current_score = score_matrix[i][j]
        if i > 0 and j > 0 and (current_score == score_matrix[i - 1][j - 1] + (
        match_score if seq1[i - 1] == seq2[j - 1] else mismatch_penalty)):
            align1 = seq1[i - 1] + align1
            align2 = seq2[j - 1] + align2
            i -= 1
            j -= 1
        elif i > 0 and (current_score == score_matrix[i - 1][j] + gap_penalty):
            align1 = seq1[i - 1] + align1
            align2 = "-" + align2
            i -= 1
        else:
            align1 = "-" + align1
            align2 = seq2[j - 1] + align2
            j -= 1

    return align1, align2, score_matrix[n][m]

In [10]:
!pip install numpy


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
result1, result2, n = needleman_wunsch("TGCA", "TGA", 1, -1, -2)
print(result1)
print(result2)

TGCA
TG-A


<h3>Faster implementation of the aligorithm using Biopython</h3>

In [14]:
from Bio import Align
sequence1 = "DVQLQASGGGLVQAGGSLRLSCAASARTFYTMGWFRQVLGKDREFVGAIRWGVYATTRYADSVKGRFSISRDDATNTVALQMNSLKPEDTAVYYCAARAGPLGFELSATSSAEYDYWGQGTQVTVSS"
sequence2 = "QVQLVESGGGLVQPGGTLRLSCAASGFTLDYYAIGWFRQAPGKEREGVSCISGSGGITNYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAPVSHTVVAGCAFEAWTDFGSWGQGTQVTVSS"

aligner = Align.PairwiseAligner() # definition of the sequence aligner
aligner.mode = 'global'
aligner.match_score = 2
aligner.mismatch_score = -5
aligner.gap_score = -2
aligner.open_gap_score = -1.0   # stronger open penalty reduces ambiguity
aligner.extend_gap_score = -0.5  # explicit extension cost also helps

alignment_res = aligner.align(sequence1, sequence2)
print(alignment_res[0])
print(alignment_res[0].score)

# Fetching out the aligned sequences
print(alignment_res[0][0])
print(alignment_res[0][1])



target            0 -DVQL--QASGGGLVQ-AGG-SLRLSCAAS-ARTF----Y--TMGWFRQ--VLGK-DRE-
                  0 --|||----|||||||--||--||||||||----|----|----|||||----||--||-
query             0 Q-VQLVE--SGGGLVQP-GGT-LRLSCAASG---FTLDYYAI--GWFRQAP--GKE-REG

target           44 FV-------GAIRWGVYAT--TRYADSVKGRF-SISRD-DA-TNTV-ALQMNSLKPEDTA
                 60 -|-------|-|------|--|---|||||||--||||--|--|||--||||||||||||
query            47 -VSCISGSGG-I------TNYT---DSVKGRFT-ISRDN-AK-NTVY-LQMNSLKPEDTA

target           91 VYYCA-------ARAGPLG--FELSA-T---SSAEYDYWGQGTQVTVSS 127
                120 |||||-------|-----|--||--|-|---|------||||||||||| 169
query            92 VYYCAPVSHTVVA-----GCAFE--AWTDFGS------WGQGTQVTVSS 128

111.0
-DVQL--QASGGGLVQ-AGG-SLRLSCAAS-ARTF----Y--TMGWFRQ--VLGK-DRE-FV-------GAIRWGVYAT--TRYADSVKGRF-SISRD-DA-TNTV-ALQMNSLKPEDTAVYYCA-------ARAGPLG--FELSA-T---SSAEYDYWGQGTQVTVSS
Q-VQLVE--SGGGLVQP-GGT-LRLSCAASG---FTLDYYAI--GWFRQAP--GKE-REG-VSCISGSGG-I------TNYT---DSVKGRFT-ISRDN-AK-NTVY-LQMNS

In [4]:
print(type(alignment_res[0]))

<class 'Bio.Align.Alignment'>


<p><b>Creating the multiple sequence alignment object and plotting it</b></p>
<a href = "https://github.com/moshi4/pyMSAviz">PymsaViz github repository</a>

In [5]:
import matplotlib.pyplot as plt
from Bio.Align import MultipleSeqAlignment
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from pymsaviz import MsaViz

# Lets's suppose we have the dictionary 
dict_sequences = {"Sequence1" : "-DVQL--QASGGGLVQ-AGG-SLRLSCAAS-ARTF----Y--TMGWFRQ--VLGK-DRE-FV-------GAIRWGVYAT--TRYADSVKGRF-SISRD-DA-TNTV-ALQMNSLKPEDTAVYYCA-------ARAGPLG--FELSA-T---SSAEYDYWGQGTQVTVSS", 
 "Sequence2" : "Q-VQLVE--SGGGLVQP-GGT-LRLSCAASG---FTLDYYAI--GWFRQAP--GKE-REG-VSCISGSGG-I------TNYT---DSVKGRFT-ISRDN-AK-NTVY-LQMNSLKPEDTAVYYCAPVSHTVVA-----GCAFE--AWTDFGS------WGQGTQVTVSS"}
alignment_list = []
for key, value in dict_sequences.items():
    a = SeqRecord(Seq(value), id=key)
    alignment_list.append(a)
align_for_plotting = MultipleSeqAlignment(alignment_list)
msa_res = MsaViz(align_for_plotting, wrap_length=100, show_grid=True, show_consensus=True, color_scheme='Clustal')
msa_res.savefig("my_figure.png")

<h3>Using the alignment to extract the framework</h3>

In [15]:
from Bio import Align
sequence = "EVQLQASGGGLVQAGGSLRLACALSGSTININTMAWYRQPPGKQREFVAILLNGGTSNYADSVTGRFTISRDNAKNTVYLQMNSLKVEDTAVYYCHLGGLGASYWGQGTQVTVSS"
reference_framework = "EVQLQASGGGLVQAGGSLRLSCAASG"

aligner = Align.PairwiseAligner() # definition of the sequence aligner
aligner.mode = 'global'
aligner.match_score = 2
aligner.mismatch_score = -2
aligner.gap_score = -2
aligner.open_gap_score = -1.0   
aligner.extend_gap_score = -0.5  

# 1. Cutting the framework in the fragments of the same length as the reference framework
ngrams = [sequence[i:i+len(reference_framework)] for i in range(len(sequence) - len(reference_framework))]

Best_score = 0
Best_sequence = None

for seq in ngrams:
    result_alignment = aligner.align(seq, reference_framework)
    score = result_alignment[0].score
    if score > Best_score:
        Best_score = score
        Best_sequence = seq
print(Best_sequence)

EVQLQASGGGLVQAGGSLRLACALSG


<h3>What to do with the partial frameworks?</h3>

<p>Framework 1 can be truncated from the left side, framework 4 from the right side.</p>

In [16]:
def extract_framework_partial(reference, sequence, threshold, direction):
    # this function is specifically designed to handle the cases when the framework is truncated.
    # this can only happen with framework 1 or framework 4
    # right should be used for framework 1
    # left for framework 4
    to_return = "Not Found"
    # first lets try to extract the full framework
    if sequence.find(reference) != -1:
        print(sequence.find(reference))
        to_return = reference
    else:
        if direction == "right":
            references = [reference[y:] for y in range(0, len(reference)-5)]
            print(references)
        elif direction == "left":
            references = [reference[:-y] for y in range(1, len(reference)-3)]
        # making the dictionary in which the refernce fragments will be attached to their lengths
        length_list = [len(reference_inst) for reference_inst in references]
        dict_length_refs = dict(zip(length_list, references))

        present = [seq_frag for seq_frag in references if sequence.find(seq_frag) != -1]
        if len(present) > 0:
            # the longest fragment becomes selected.
            to_return = sorted(present, key=lambda seq_frag: len(seq_frag), reverse=True)[0]
        else:
            best_framework = None
            best_score = 0
            seq_fragments_refrences = []
            # generate all possible fragment in case multiple references are used
            for seq_frag_ref in references:
                seq_fragments_ref_for_one_length = [sequence[i:i + len(seq_frag_ref)]
                             for i in range(0, len(sequence) - len(seq_frag_ref))]
                seq_fragments_refrences.extend(seq_fragments_ref_for_one_length)

            for seq_fragment in seq_fragments_refrences:
                # fetching out the correct reference
                correct_reference = dict_length_refs[len(seq_fragment)]
                result = sorted(aligner.align(seq_fragment, correct_reference))
                if result[0].score > best_score:
                    best_score = result[0].score
                    best_framework = seq_fragment

            if best_framework is not None:
                if best_score > threshold*len(best_framework):
                    to_return = best_framework

    return to_return

sequence1 = "GGGLVQAGGSLRLACALSGSTININTMAWYRQPPGKQREFVAILLNGGTSNYADSVTGRFTISRDNAKNTVYLQMNSLKVEDTAVYYCHLGGLGASYWGQGTQVTVSS"
reference_framework = "EVQLQASGGGLVQAGGSLRLSCAASG"
print(extract_framework_partial(reference_framework, sequence1, 0.6, "right"))
      


['EVQLQASGGGLVQAGGSLRLSCAASG', 'VQLQASGGGLVQAGGSLRLSCAASG', 'QLQASGGGLVQAGGSLRLSCAASG', 'LQASGGGLVQAGGSLRLSCAASG', 'QASGGGLVQAGGSLRLSCAASG', 'ASGGGLVQAGGSLRLSCAASG', 'SGGGLVQAGGSLRLSCAASG', 'GGGLVQAGGSLRLSCAASG', 'GGLVQAGGSLRLSCAASG', 'GLVQAGGSLRLSCAASG', 'LVQAGGSLRLSCAASG', 'VQAGGSLRLSCAASG', 'QAGGSLRLSCAASG', 'AGGSLRLSCAASG', 'GGSLRLSCAASG', 'GSLRLSCAASG', 'SLRLSCAASG', 'LRLSCAASG', 'RLSCAASG', 'LSCAASG', 'SCAASG']
GGGLVQAGGSLRLACALSG
